# Data complexity effects on synthetic data quality

## Step 2: Fit synthesizers and generate samples

This notebook reads the source-data manifest from step 1, fits one neural synthesizer and one tree-based model, saves each model, writes synthetic CSVs, and records them in a synthetic manifest.

## Load libraries

In [2]:
!pip install sdv
!pip install python-synthpop

In [3]:
import os
import random
from pathlib import Path
import numpy as np
import pandas as pd
import torch

from sdv.metadata import Metadata
from sdv.single_table import TVAESynthesizer
from synthpop import MissingDataHandler, DataProcessor, CARTMethod
import pickle


In [4]:
EXP = "syndaite"

PROJECT_DIR = Path.cwd().resolve()

def project_path(path):
  path = Path(path)
  return path if path.is_absolute() else PROJECT_DIR / path

def relative_to_project(path):
  return str(project_path(path).resolve().relative_to(PROJECT_DIR))


INPUT_DIR = PROJECT_DIR / "data" / "synth_input" / EXP
OUTPUT_DIR = PROJECT_DIR / "data" / "synth_output" / EXP
MODEL_DIR = PROJECT_DIR / "data" / "synth_models" / EXP
RESULT_DIR = PROJECT_DIR / "results" / EXP
FIG_DIR = PROJECT_DIR / "results" / "figs" / EXP

RANDOM_SEEDS = [42]
NREPS = 3

REQUIRE_GPU = True

In [6]:
GPU_AVAILABLE = torch.cuda.is_available()

if REQUIRE_GPU and not GPU_AVAILABLE:
  raise RuntimeError("No GPU is available. Please set REQUIRE_GPU to False to run on CPU.")

if GPU_AVAILABLE:
  print("Using CUDA GPU:", torch.cuda.get_device_name(0))
else:
  print("No CUDA GPU detected; SDV will train on CPU.")

In [7]:
def fix_target_dtype(df, target_col):
  df[target_col] = df[target_col].astype("object")
  return(df)


def make_gpu_synthesizer(model_class, meta, model_kwargs):
  gpu_kwargs = dict(model_kwargs)
  gpu_kwargs["enable_gpu"] = True

  try:
    return model_class(meta, **gpu_kwargs)
  except TypeError as err:
    if "enable_gpu" not in str(err):
      raise

    gpu_kwargs.pop("enable_gpu")
    gpu_kwargs["cuda"] = True
    return model_class(meta, **gpu_kwargs)


## Load source manifest

In [8]:
manifest = pd.read_csv(INPUT_DIR / f"{EXP}_manifest.csv")
manifest

## Fit TVAE synthesizer and sample

In [9]:
# Set seeds
np.random.seed(RANDOM_SEEDS[0])
torch.manual_seed(RANDOM_SEEDS[0])
if GPU_AVAILABLE:
  torch.cuda.manual_seed_all(RANDOM_SEEDS[0])
random.seed(RANDOM_SEEDS[0])

In [10]:
for path in [OUTPUT_DIR, MODEL_DIR]:
  path.mkdir(parents=True, exist_ok=True)

nn_models = {
    "TVAE": (TVAESynthesizer, {})
}

synthetic_rows = []
model_rows = []

i=0
for dataset in manifest.itertuples(index=False):

  if(i % 100 == 0): 
     print(dataset.Dataset)
     
  train = pd.read_csv(project_path(dataset.train_path))
  target = dataset.target_col
  train = fix_target_dtype(train, target_col=target)
  meta = Metadata.load_from_json(str(project_path(dataset.metadata_path)))

  for model_name, model_info in nn_models.items():
    
    model_class, model_kwargs = model_info

    model_syn = make_gpu_synthesizer(model_class, meta, model_kwargs)
    model_syn.fit(train)

    model_path = MODEL_DIR / f"{EXP}_{dataset.Dataset}_{model_name}.pkl"
    model_syn.save(str(model_path))

    model_rows.append({
        "Dataset": dataset.Dataset,
        "Model": model_name,
        "base_path": str(PROJECT_DIR),
        "real_path": dataset.train_path,
        "model_path": relative_to_project(model_path),
    })

    for rep in range(NREPS):
      rep_name = "R" + str(rep + 1)

      syn_df = model_syn.sample(num_rows=train.shape[0])
      syn_path = OUTPUT_DIR / f"{EXP}_{dataset.Dataset}_{model_name}_{rep_name}.csv"
      syn_df.to_csv(syn_path, index=False)

      synthetic_rows.append({
          "Dataset": dataset.Dataset,
          "Model": model_name,
          "Repetition": rep_name,
          "base_path": str(PROJECT_DIR),
          "real_path": dataset.train_path,
          "synthetic_path": relative_to_project(syn_path),
          "model_path": relative_to_project(model_path),
      })

    del model_syn
    i=i+1


## Fit CART synthesizer and sample

In [12]:
# Set seeds
np.random.seed(RANDOM_SEEDS[0])
random.seed(RANDOM_SEEDS[0])


i = 0
for dataset in manifest.itertuples(index=False):
  if(i % 100 == 0): 
     print(dataset.Dataset)
   
  train = pd.read_csv(project_path(dataset.train_path))
  target = dataset.target_col
  train = fix_target_dtype(train, target_col=target)

  md_handler = MissingDataHandler()
  metadata= md_handler.get_column_dtypes(train)

  processor = DataProcessor(metadata)
  processed_data = processor.preprocess(train)

  model_name = "CART"
  model_path = MODEL_DIR / f"{EXP}_{dataset.Dataset}_{model_name}.pkl"
  cart = CARTMethod(metadata, smoothing=True, proper=True, minibucket=5, random_state=RANDOM_SEEDS[0]) 
  cart.fit(processed_data)

  model_rows.append({
        "Dataset": dataset.Dataset,
        "Model": model_name,
        "base_path" : str(PROJECT_DIR),
        "real_path": dataset.train_path,
        "model_path": relative_to_project(model_path),
  })

  with open(model_path, 'wb') as file:
    pickle.dump(cart, file)

  for rep in range(NREPS):
    rep_name = "R" + str(rep + 1)

    syn_df = cart.sample(train.shape[0])
    syn_df = processor.postprocess(syn_df)

    syn_path = OUTPUT_DIR / f"{EXP}_{dataset.Dataset}_{model_name}_{rep_name}.csv"
    syn_df.to_csv(syn_path, index=False)

    synthetic_rows.append({
          "Dataset": dataset.Dataset,
          "Model": model_name,
          "Repetition": rep_name,
          "base_path" : str(PROJECT_DIR),
          "real_path": dataset.train_path,
          "synthetic_path": relative_to_project(syn_path),
          "model_path": relative_to_project(model_path),
    })
    i = i + 1


In [13]:
synthetic_manifest = pd.DataFrame(synthetic_rows)
model_manifest = pd.DataFrame(model_rows)

synthetic_manifest.to_csv(OUTPUT_DIR / f"{EXP}_synthetic_manifest.csv", index=False)
model_manifest.to_csv(MODEL_DIR / f"{EXP}_model_manifest.csv", index=False)
synthetic_manifest


## Quick checks

In [14]:
for row in synthetic_manifest.itertuples(index=False):
  syn_df = pd.read_csv(project_path(row.synthetic_path))
  print(row.Dataset, row.Model, row.Repetition, syn_df.shape, syn_df["target"].value_counts().to_dict())
